# W12 Practice — An MCP Server of Our Own

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ralbu85/stml_2026/blob/main/lectures/week12/W12_practice_mcp_server.ipynb)

**Goal.** Build an MCP server over a web API, use it from an agent, and define its tools by two principles of tool design. One question runs on two servers over the same API: first a server that wraps each request of the API as a tool, then a server whose tools follow the principles.

| Part | What you learn | Main names |
|---|---|---|
| 1. The recipe API | two requests to a web API | `httpx2` |
| 2. A server that wraps the API | one tool for each request; the tools in an agent | `MCPServer`, `@mcp.tool`, `MCPAdapter` |
| 3. Two principles of tool design | what a model needs from a tool | |
| 4. A server built by the principles | two tools for the same question, with typed results | `BaseModel` |
| 5. Exercises | three more tools | |

**Cell labels.** The label is the first line of a code cell, or the second line when the cell saves a file with `%%writefile`:

- `# ▶ RUN · example`: read each line, then run.
- `# ▶ RUN · setup`: just run.
- `# ✍️ TRY · exercise`: write your code in the cell, then run (Part 5).

## An MCP Server over a Web API

An MCP server gives a web API the form of tools: each tool is a function that sends a request to the API and returns the reply. Three programs take part.

| Program | What it does | Where |
|---|---|---|
| Web API | keeps the recipes and answers HTTP requests | TheMealDB, on the internet |
| MCP server | offers tools; each tool calls the web API | a file in this runtime |
| MCP host | gives the tools of the server to an agent | this notebook |

A server in Python is written in five steps.

| Step | Code |
|---|---|
| 1. Make the server object | `mcp = MCPServer("recipes")` |
| 2. Write one function for each tool | `@mcp.tool()` above the function |
| 3. Call the web API in the function and return the result | `httpx2.get(...)` |
| 4. Run the server | `mcp.run()` |
| 5. Connect a client and get the tools | `MCPAdapter(PythonStdioTransport("server.py"))`, `list_tools()` |

The server makes the definition of a tool from the function: the function name becomes the tool name, the type hints the arguments, and the docstring the description. The client starts the server file as a program and talks to it through its standard input and output (the **stdio** transport of MCP).

Source of the steps: MCP documentation, *Build an MCP server*.

## 0. Setup

*Do:* run the two cells; paste your API key into the second.

In [ ]:
# ▶ RUN · setup: just run
%pip install -q "langchain[mcp]==1.4.4" "langchain-openai==1.7.0" "langgraph==1.2.14"

In [ ]:
# ▶ RUN · setup: just run
import os
from langchain.chat_models import init_chat_model

OPENAI_API_KEY = "PASTE-YOUR-KEY-HERE"   # paste your key between the quotes
if OPENAI_API_KEY != "PASTE-YOUR-KEY-HERE":
    os.environ["OPENAI_API_KEY"] = OPENAI_API_KEY   # LangChain reads the key from here

MODEL = "openai:gpt-5.4"
model = init_chat_model(MODEL)                     # one model object for the whole notebook

## 1. The Recipe API

TheMealDB is a free database of recipes with a web API that needs no sign-up. Two of its requests are used first.

| Request | What it returns |
|---|---|
| `GET /filter.php?i=<ingredient>` | the meals that use one ingredient |
| `GET /lookup.php?i=<meal ID>` | the recipe of one meal |

*Do:* run the cell. It sends both requests and prints the replies.

In [ ]:
# ▶ RUN · example: read each line, then run
import httpx2                                      # an HTTP client for Python; it came with the packages of the setup

API = "https://www.themealdb.com/api/json/v1/1"    # the address of the API; "1" is the public test key

reply = httpx2.get(API + "/filter.php", params={"i": "chicken breast"})   # the meals that use chicken breast
meals = reply.json()["meals"]                      # .json() turns the JSON of the reply into Python lists and dicts
print(len(meals), "meals; the first one:", meals[0])

reply = httpx2.get(API + "/lookup.php", params={"i": meals[0]["idMeal"]})   # the recipe of the first meal, by its ID
print(reply.text)

**Reading the output.** A search returns the name, a picture link, the ID, and the country of each meal, for one ingredient only. A recipe needs a second request; its reply has 54 fields, among them twenty pairs `strIngredient1` to `strIngredient20` and `strMeasure1` to `strMeasure20`, and the pairs that a meal does not use are empty.

## 2. A Server that Wraps the API

The first server has one tool for each request and returns the reply of the API as it comes.

*Do:* run the cell. It saves the server as `meal_mcp_v1.py`.

In [ ]:
%%writefile meal_mcp_v1.py
# ▶ RUN · example: read each line, then run
import httpx2
from mcp.server import MCPServer

API = "https://www.themealdb.com/api/json/v1/1"

## Step 1 - the server object
mcp = MCPServer("recipes", log_level="WARNING")    # the name that the server gives to its clients; log only warnings


## Steps 2 and 3 - one function for each tool; the function calls the web API
@mcp.tool()
def search_by_ingredient(ingredient: str) -> str:
    """Find the meals that use one ingredient.

    Args:
        ingredient: One ingredient, for example "chicken breast".
    """
    return httpx2.get(API + "/filter.php", params={"i": ingredient}).text   # the reply of the API is the tool result


@mcp.tool()
def get_recipe(meal_id: str) -> str:
    """Get the recipe of one meal.

    Args:
        meal_id: The ID of the meal, for example "52772".
    """
    return httpx2.get(API + "/lookup.php", params={"i": meal_id}).text


## Step 4 - run the server
if __name__ == "__main__":                         # true when this file is started as a program
    mcp.run()                                      # stdio: read requests from standard input, write replies to standard output

### 2.2 The question on this server

*Do:* read `QUESTION`, then run the cell. It connects to the server (step 5), gives its tools to an agent, and prints the path of the run: each tool call, the length of each tool result, and the answer.

In [ ]:
# ▶ RUN · example: read each line, then run
from pathlib import Path
from fastmcp.client.transports import PythonStdioTransport
from langchain.mcp import MCPAdapter               # this import prints one warning: this part of LangChain is in beta
from langchain.agents import create_agent
from langchain.messages import HumanMessage

def show(result):
    """Print the path of one run: the tool calls, the length of each tool result, and the answer."""
    for message in result["messages"]:
        if message.type == "ai":
            for call in message.tool_calls:        # one AI message can ask for several tools
                print("TOOL CALL  :", call["name"], call["args"])
        if message.type == "tool":
            print("TOOL RESULT:", message.name, "-", len(message.text), "characters")
    print()
    print(result["messages"][-1].text)             # the last message is the answer

## Step 5 - connect a client and get the tools
server_v1 = PythonStdioTransport("meal_mcp_v1.py", log_file=Path("server.log"))   # start the file as a program, talk over stdio;
                                                   # its messages go to server.log (Colab has no place to show them)
async with MCPAdapter(server_v1) as adapter:
    tools_v1 = await adapter.list_tools()

agent_v1 = create_agent(model, tools_v1)

QUESTION = "닭가슴살과 마늘이 모두 들어가는 요리 두 개를 골라 재료와 조리 순서를 알려줘."
result = await agent_v1.ainvoke({"messages": [HumanMessage(QUESTION)]})   # the tools are asynchronous, so the agent runs with ainvoke
show(result)

**Reading the output.** In our run the model made five calls. The search tool takes one ingredient, so the model searched for chicken breast only and then asked for four recipes to see which of them use garlic; it answered with two of them. The search returned 3,323 characters, with a picture link for each meal, and each recipe 1,845 to 3,173 characters, with the empty fields; the model read 13,615 characters in all.

## 3. Two Principles of Tool Design

A request of an API is designed for a program, which reads any amount of JSON at once. A tool is used by a model, which reads every result as text in a context of limited size. Anthropic's guide for writers of tools starts from this difference; two of its principles concern the server of Part 2.

| Principle | What it asks | The server of Part 2 | The server of Part 4 |
|---|---|---|---|
| 1. Tools for the task, not for each request of the API | a tool does one step that the user wants done; it can send several requests to the API in one call, so that the model does not chain the requests and read their intermediate results | a search for one ingredient; to find the meals that also use garlic, the model asked for four recipes and read them | `find_by_ingredients` takes several ingredients, sends one request for each, and keeps the meals that are in every list |
| 2. Results with only what the next step needs | the result holds the fields that the model uses next, without low-level fields such as picture links, and the number of results has a limit | the JSON of the API: a picture link for each meal, 54 fields for a recipe, the empty ones too | name, country, and ID of each meal, at most `limit` meals, with `count`; a recipe with its ingredients and instructions, without the empty fields |

Source: Anthropic, *Writing effective tools for agents* (2025), sections "Choosing the right tools for agents", "Returning meaningful context from your tools", and "Optimizing tool responses for token efficiency".

## 4. A Server Built by the Principles

The second server has two tools for the same question, and its descriptions are written as in Part 2. Its search tool takes all the ingredients at once (principle 1). Each tool returns a Pydantic object with only the fields that the model uses next (principle 2); the server declares the class as the output schema of the tool and sends the object as structured content, which the model reads as JSON.

*Do:* run the cell. It saves the server as `meal_mcp_v2.py`.

In [ ]:
%%writefile meal_mcp_v2.py
# ▶ RUN · example: read each line, then run
import httpx2
from pydantic import BaseModel
from mcp.server import MCPServer

API = "https://www.themealdb.com/api/json/v1/1"
mcp = MCPServer("recipes", log_level="WARNING")


def get_meals(path, params):
    """Send one request to the API and return its list of meals ([] when there is none). Not a tool."""
    meals = httpx2.get(API + path, params=params).json()["meals"]
    if meals is None:
        return []
    return meals


## Principle 2 - the result classes: only the fields that the next step needs
class Meal(BaseModel):
    name: str
    country: str | None
    meal_id: str                                   # the next step, get_recipe, needs it


class MealList(BaseModel):
    count: int                                     # the number of all meals found
    meals: list[Meal]                              # at most `limit` of them


class Ingredient(BaseModel):
    name: str
    measure: str


class Recipe(BaseModel):
    name: str
    ingredients: list[Ingredient]                  # without the empty pairs of the API
    instructions: str


def short(meal):
    """A Meal from a meal of the API: name, country, and ID; the picture link is left out. Not a tool."""
    return Meal(name=meal["strMeal"], country=meal["strCountry"], meal_id=meal["idMeal"])


## Principle 1 - one tool for the search of the user: all the ingredients in one call
@mcp.tool()
def find_by_ingredients(ingredients: list[str], limit: int = 5) -> MealList:
    """Find the meals that use all of the given ingredients.

    Args:
        ingredients: Ingredients in English, for example ["chicken breast", "garlic"].
        limit: The largest number of meals to return.
    """
    found = None
    for ingredient in ingredients:                 # the API takes one ingredient per request
        meals = get_meals("/filter.php", {"i": ingredient})
        if found is None:
            found = meals
        else:
            ids = [meal["idMeal"] for meal in meals]
            found = [meal for meal in found if meal["idMeal"] in ids]   # keep the meals that are in both lists
    return MealList(count=len(found), meals=[short(meal) for meal in found[:limit]])


## Principle 2 - the recipe without the empty fields
@mcp.tool()
def get_recipe(meal_id: str) -> Recipe:
    """Get the recipe of one meal.

    Args:
        meal_id: The ID of the meal, for example "52772".
    """
    meal = get_meals("/lookup.php", {"i": meal_id})[0]
    ingredients = []
    for i in range(1, 21):                         # the API keeps twenty numbered pairs
        name = meal["strIngredient" + str(i)]
        if name:                                   # skip the empty pairs
            ingredients.append(Ingredient(name=name, measure=str(meal["strMeasure" + str(i)])))
    return Recipe(name=meal["strMeal"], ingredients=ingredients, instructions=meal["strInstructions"])


if __name__ == "__main__":
    mcp.run()

### 4.2 The same question on this server

*Do:* run the cell. It connects to the second server, prints its tools, and runs `QUESTION` again.

In [ ]:
# ▶ RUN · example: read each line, then run
server_v2 = PythonStdioTransport("meal_mcp_v2.py", log_file=Path("server.log"))
async with MCPAdapter(server_v2) as adapter:
    tools_v2 = await adapter.list_tools()

for tool in tools_v2:
    print(tool.name, list(tool.args))
print()

agent_v2 = create_agent(model, tools_v2)
result = await agent_v2.ainvoke({"messages": [HumanMessage(QUESTION)]})
show(result)

**Reading the output.** In our run the model made three calls. One call of `find_by_ingredients` found the meals that use both ingredients; the model set `limit` to 2 because the question asks for two meals. It then asked for the two recipes in one reply. The search returned 261 characters and the recipes 2,496 and 1,251; the model read 4,008 characters in all, against 13,615 in Part 2, and asked for no recipe that it did not use.

## 5. Exercises: Three More Tools

Each exercise adds one tool for a question that the two tools do not answer well. Write the tools in `meal_mcp_v3.py`. Its first lines import the server of Part 4 and its result classes, so your tools join its two tools. Follow the two principles: a tool for the task of the question, and a result with only what the next step needs.

### 5.1 `find_by_country`

Question: 일본 요리 중에 닭가슴살이 들어가는 요리를 알려줘. The API returns the meals of a country with `GET /filter.php?a=<country>`, the country as an English adjective such as `Japanese`.

*Do:* write a tool that takes a country and, optionally, one ingredient, and returns the meals of that country that use the ingredient, as a `MealList`.

In [ ]:
%%writefile meal_mcp_v3.py
# ✍️ TRY · exercise: write your code in this cell, then run
from pydantic import BaseModel
from meal_mcp_v2 import mcp, get_meals, short, MealList   # the server of Part 4 with its two tools

# find_by_country(country, ingredient, limit) -> MealList: the meals of one country, only those with the ingredient if one is given

### 5.2 `search_by_name`

Question: 카레 요리를 찾아줘. The API returns the meals whose name contains a word with `GET /search.php?s=<word>`, each with all 54 fields.

*Do:* write a tool that takes a word of a meal name and returns a `MealList` with at most `limit` meals.

In [ ]:
%%writefile -a meal_mcp_v3.py
# ✍️ TRY · exercise: write your code in this cell, then run

# search_by_name(name, limit) -> MealList: the meals whose name contains the word

### 5.3 `check_ingredient`

Question: 대파랑 닭가슴살이 들어가는 요리 있어? The English name of 대파 is not clear, and the database knows 992 ingredient names, which `GET /list.php?i=list` returns.

*Do:* write a tool that takes an ingredient name and returns whether the database knows it and, if not, similar names that it knows. Define a result class for it.

In [ ]:
%%writefile -a meal_mcp_v3.py
# ✍️ TRY · exercise: write your code in this cell, then run

# class IngredientCheck(BaseModel): known, similar
# check_ingredient(name) -> IngredientCheck: is the name in the database? if not, which similar names are?

### 5.4 Your tools in the agent

*Do:* run the two cells. The first ends the file with the lines that run the server; the second connects to it, prints its tools, and runs the three questions.

In [ ]:
%%writefile -a meal_mcp_v3.py
# ▶ RUN · setup: just run


if __name__ == "__main__":
    mcp.run()

In [ ]:
# ▶ RUN · example: read each line, then run
server_v3 = PythonStdioTransport("meal_mcp_v3.py", log_file=Path("server.log"))
async with MCPAdapter(server_v3) as adapter:
    tools_v3 = await adapter.list_tools()

for tool in tools_v3:
    print(tool.name, list(tool.args))

agent_v3 = create_agent(model, tools_v3)
TESTS = [
    "일본 요리 중에 닭가슴살이 들어가는 요리를 알려줘.",
    "카레 요리를 찾아줘.",
    "대파랑 닭가슴살이 들어가는 요리 있어?",
]
for test in TESTS:
    print()
    print("=====", test)
    result = await agent_v3.ainvoke({"messages": [HumanMessage(test)]})
    show(result)